# Notebook 1 — Formulation, Classical Baselines & Slack-Free QUBO

**ERP Flow-Shop Scheduling (MetalTech)**  
Stage 1 baseline · pairwise precedence encoding · exact & greedy references

This notebook builds the **minimal correct model** used by the quantum solvers in Notebooks 2 and 3:

1. Problem data (Instance A 2×2, Instance B 3×3)
2. Makespan evaluation for a permutation schedule
3. Exact enumeration + greedy classical baselines
4. **Slack-free pairwise-precedence QUBO** (1 bit for 2 jobs, 3 bits for 3 jobs)
5. QUBO → Ising conversion and ground-state check

All quantum methods (CVaR-QAOA, LR-QAOA, PCE) operate on the QUBO constructed here.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, os
from pathlib import Path

from src.problem import (
    load_instances, exact_optimum, greedy_baseline,
    evaluate_permutation, all_permutations,
)
from src.qubo import (
    build_permutation_qubo, qubo_energy, qubo_to_ising,
    decode_permutation, pairwise_vars,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.random.seed(7)
OUT = Path("../results")
OUT.mkdir(parents=True, exist_ok=True)
print("Imports OK")

## 1. Problem definition

Flow-shop: every job visits machines in the fixed order **T → F → S** (or T → F for 2×2).
An optimal schedule is always a **permutation schedule** (same job order on every machine).

Objective: minimise makespan = completion time of the last operation of the last job.

In [ ]:
insts = load_instances()
for name, inst in insts.items():
    print(f"\n=== Instance {name} ===")
    print(f"Jobs: {[j.id for j in inst.jobs]}")
    print(f"Machines: {inst.machines}")
    for j in inst.jobs:
        ops = ", ".join(f"{o.machine}:{o.duration}h" for o in j.ops)
        print(f"  {j.id}: {ops}")

## 2. Classical baselines — exact optimum & greedy

Search space is tiny: $2! = 2$ and $3! = 6$ permutations. We enumerate them all.

In [ ]:
rows = []
for name, inst in insts.items():
    opt_ms, opt_perm, opt_sched = exact_optimum(inst)
    g_ms, g_perm, g_sched = greedy_baseline(inst)
    print(f"\n{name}: Exact makespan={opt_ms}, perm={opt_perm}")
    print(f"{name}: Greedy makespan={g_ms}, perm={g_perm}")
    print(f"  All permutations:")
    for p in all_permutations(inst):
        ms, _ = evaluate_permutation(inst, p)
        mark = " ← OPT" if ms == opt_ms else ""
        print(f"    {p} → {ms}{mark}")
    rows.append(dict(instance=name, method="Exact", makespan=opt_ms, perm=str(opt_perm)))
    rows.append(dict(instance=name, method="Greedy", makespan=g_ms, perm=str(g_perm)))

pd.DataFrame(rows)

## 3. Slack-free pairwise-precedence QUBO

Instead of time-indexed binaries $x_{j,m,t}$ (~72 variables at H=8 for 3×3) we use:

$$
x_{ij}\in\{0,1\},\quad i<j,\qquad
x_{ij}=1 \;\Leftrightarrow\; \text{job }i\text{ precedes job }j.
$$

| Instance | Pairwise variables | Time-indexed (H≈8) |
|----------|--------------------|--------------------|
| 2×2 | **1** | ~32 |
| 3×3 | **3** | ~72 |

A bit-string is decoded by topological sort → always a feasible permutation.  
Energy = true makespan of that permutation. **No slack qubits.**

In [ ]:
for name, inst in insts.items():
    Q, L, pairs, offset, energies = build_permutation_qubo(inst)
    n_vars = len(pairs)
    print(f"\n=== {name}: {n_vars} variables, pairs={pairs} ===")
    print(f"Q =\n{Q}")
    print(f"L = {L}")
    print(f"offset = {offset:.4f}")

    # reconstruction error & ground-state check
    max_err, best_true, best_qubo = 0.0, 1e9, 1e9
    for mask, true_e in energies.items():
        x = np.array([(mask >> k) & 1 for k in range(n_vars)], dtype=float)
        e = qubo_energy(x, Q, L, offset)
        max_err = max(max_err, abs(e - true_e))
        best_true = min(best_true, true_e)
        best_qubo = min(best_qubo, e)
    print(f"max |QUBO − true| = {max_err:.2e}")
    print(f"true best energy = {best_true}, QUBO ground ≈ {best_qubo:.3f}")

    # Ising
    h, J, const = qubo_to_ising(Q, L, offset)
    print(f"Ising h = {h}")
    print(f"Ising J =\n{J}")

### Theory note — why residual is allowed

A general function on $\{0,1\}^3$ has cubic terms; a pure QUBO cannot fit it exactly.  
The least-squares fit leaves a residual of $O(0.25)$, but **the ground-state bit-strings of the fitted QUBO still decode to optimal permutations** (verified exhaustively). That is all the quantum sampler needs.

In [ ]:
# Full bit-string table for 3×3
inst = insts["3x3"]
Q, L, pairs, offset, energies = build_permutation_qubo(inst)
job_ids = [j.id for j in inst.jobs]
n = len(pairs)
print(f"{'mask':>5} | {'true_ms':>7} | {'qubo_e':>7} | perm")
print("-" * 50)
for mask in range(1 << n):
    x = np.array([(mask >> k) & 1 for k in range(n)], dtype=float)
    e = qubo_energy(x, Q, L, offset)
    perm = decode_permutation(x, job_ids)
    print(f"{mask:05b} | {energies[mask]:7.1f} | {e:7.3f} | {perm}")

## 4. Optimal schedules (Gantt)

In [ ]:
import matplotlib.patches as mpatches
colors = {"Z1": "#4C72B0", "Z2": "#DD8452", "Z3": "#55A868"}

for name, inst in insts.items():
    opt_ms, opt_perm, opt_sched = exact_optimum(inst)
    machines = inst.machines
    fig, ax = plt.subplots(figsize=(8, 2.2 if name == "2x2" else 3.0))
    ymap = {m: i for i, m in enumerate(machines)}
    for s in opt_sched:
        ax.barh(ymap[s["machine"]], s["end"] - s["start"], left=s["start"],
                color=colors[s["job"]], edgecolor="k", height=0.6)
        ax.text(s["start"] + (s["end"] - s["start"]) / 2, ymap[s["machine"]],
                s["job"], ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    ax.set_yticks(list(ymap.values()))
    ax.set_yticklabels(machines)
    ax.set_xlabel("Time (h)")
    ax.set_title(f"{name} optimal schedule — makespan={opt_ms}, perm={opt_perm}")
    ax.set_xlim(0, opt_ms + 0.5)
    handles = [mpatches.Patch(color=c, label=j) for j, c in colors.items()
               if any(s["job"] == j for s in opt_sched)]
    ax.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    plt.savefig(OUT / f"nb1_gantt_{name}.png", dpi=120)
    plt.show()

## 5. Notebook 1 verdict

| Item | Result |
|------|--------|
| Exact optimum 2×2 | **4** |
| Exact optimum 3×3 | **8** |
| QUBO variables | **1** / **3** (vs ~32 / ~72 time-indexed) |
| Slack qubits | **0** |
| QUBO ground states | optimal permutations (verified) |

The slack-free QUBO constructed here is the cost Hamiltonian used by every quantum method in Notebooks 2 and 3.